# 视频横向拼接器
每行输入一个视频路径，可选添加 annotation（最多 6 个）。

In [ ]:
import os
import re
import base64
import mimetypes
import tempfile
import subprocess
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

POSITIONS = ["top", "bottom", "top-left", "top-right", "bottom-left", "bottom-right"]
MAX_ANNOTATIONS = 6

DEFAULT_FONTCOLOR = "white"
COLOR_MAP = {
    "red":    "0xFF6B6B",
    "blue":   "0x5BC0EB",
    "yellow": "0xF7E967",
}
COLOR_HTML = {
    "red":    "#FF6B6B",
    "blue":   "#5BC0EB",
    "yellow": "#F7E967",
}
_COLOR_TOKEN_RE = re.compile(r"^\\(red|blue|yellow)\s*")
_SRT_TOKEN_RE = re.compile(r"^\\srt\s+(.+)$")


def parse_color_prefix(line: str):
    """取出行首的 \\red / \\blue / \\yellow 颜色前缀，返回 (剩余文本, 颜色名 或 None)。"""
    m = _COLOR_TOKEN_RE.match(line)
    if not m:
        return line, None
    return line[m.end():], m.group(1)


def color_to_ass(color: str):
    """将颜色名转换为 ASS/libass 的 &HAABBGGRR 颜色值（用于字幕 force_style）。"""
    hex_rgb = COLOR_MAP.get(color)
    if not hex_rgb:
        return None
    rgb = hex_rgb[2:]  # 去掉 0x 前缀 -> RRGGBB
    r, g, b = rgb[0:2], rgb[2:4], rgb[4:6]
    return f"&H00{b}{g}{r}"


POSITION_EXPR = {
    "top":          "x=(w-text_w)/2:y=30",
    "bottom":       "x=(w-text_w)/2:y=h-text_h-30",
    "top-left":     "x=30:y=30",
    "top-right":    "x=w-text_w-30:y=30",
    "bottom-left":  "x=30:y=h-text_h-30",
    "bottom-right": "x=w-text_w-30:y=h-text_h-30",
}

# 字幕位置映射到 subtitles 滤镜 force_style 的 Alignment。
# 注意：这里用的是旧版 SSA 编号（非 ASS \an 小键盘编号），经实测：
#   1/2/3 = 下 左/中/右，5/6/7 = 上 左/中/右，9/10/11 = 中 左/中/右。
POSITION_ASS_ALIGN = {
    "top":          6,
    "bottom":       2,
    "top-left":     5,
    "top-right":    7,
    "bottom-left":  1,
    "bottom-right": 3,
}

FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
    "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/dejavu/DejaVuSans.ttf",
]
FONT_PATH = next((f for f in FONT_CANDIDATES if os.path.isfile(f)), None)

SUPPORTED_EXTS = (".mp4",)
SUBTITLE_EXTS = (".srt", ".ass", ".ssa", ".vtt")

_DRAWTEXT_ESCAPE = str.maketrans({
    "\\": r"\\",
    ":": r"\:",
    "'": r"\'",
    "%": r"\%",
})


def validate_path(path: str):
    if not path:
        return None, "路径为空"
    path = os.path.realpath(os.path.expanduser(path.strip()))
    if not path.lower().endswith(SUPPORTED_EXTS):
        return None, f"仅支持 .mp4: {path}"
    if not os.path.isfile(path):
        return None, f"文件不存在: {path}"
    return path, ""


def validate_subtitle_path(path: str):
    if not path:
        return None, "字幕路径为空"
    path = os.path.realpath(os.path.expanduser(path.strip()))
    if not path.lower().endswith(SUBTITLE_EXTS):
        return None, f"仅支持字幕文件 {'/'.join(SUBTITLE_EXTS)}: {path}"
    if not os.path.isfile(path):
        return None, f"字幕文件不存在: {path}"
    return path, ""


def has_audio_stream(path: str) -> bool:
    try:
        out = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-select_streams", "a",
                "-show_entries", "stream=index",
                "-of", "csv=p=0",
                path,
            ],
            capture_output=True, text=True, timeout=15,
        )
        return bool(out.stdout.strip())
    except Exception:
        return False


def escape_drawtext(text: str) -> str:
    return text.translate(_DRAWTEXT_ESCAPE)


def escape_filter_path(path: str) -> str:
    """转义文件路径，供 filtergraph 单引号内的 filename= 使用。"""
    return (
        path.replace("\\", "\\\\")
            .replace(":", "\\:")
            .replace("'", "\\'")
    )


def expand_annotation(text: str, video_path: str) -> str:
    """将 \\frame_idx \\name \\timestamp 转义命令展开为 drawtext text= 可用的片段。"""
    name_val = escape_drawtext(os.path.basename(video_path))
    frame_expr = "%{eif\\:n\\:d}"
    # 秒表：显示当前播放时间戳 HH\:MM\:SS.mmm（drawtext 内冒号需转义）。
    timestamp_expr = "%{pts\\:hms}"

    tokens = re.split(r"(\\frame_idx|\\name|\\timestamp)", text)
    out_parts = []
    for tok in tokens:
        if tok == "\\name":
            out_parts.append(name_val)
        elif tok == "\\frame_idx":
            out_parts.append(frame_expr)
        elif tok == "\\timestamp":
            out_parts.append(timestamp_expr)
        else:
            out_parts.append(escape_drawtext(tok))
    return "".join(out_parts)


def build_drawtext(expanded_text: str, position: str, color: str = None) -> str:
    font_opt = f":fontfile={FONT_PATH}" if FONT_PATH else ""
    fontcolor = COLOR_MAP.get(color, DEFAULT_FONTCOLOR)
    return (
        f"drawtext=text='{expanded_text}'"
        f"{font_opt}"
        f":fontcolor={fontcolor}:fontsize=28"
        f":box=1:boxcolor=black@0.5:boxborderw=8"
        f":{POSITION_EXPR[position]}"
    )


def build_subtitles(srt_path: str, position: str = "bottom", color: str = None) -> str:
    """用 libass 的 subtitles 滤镜把字幕文件按时间轴烧录进画面；
    位置由 position（对应下拉框）映射到 force_style 的 Alignment，颜色覆盖主色。"""
    escaped = escape_filter_path(srt_path)
    style_parts = [f"Alignment={POSITION_ASS_ALIGN.get(position, 2)}"]
    ass_color = color_to_ass(color)
    if ass_color:
        style_parts.append(f"PrimaryColour={ass_color}")
    style_opt = f":force_style='{','.join(style_parts)}'"
    return f"subtitles=filename='{escaped}'{style_opt}"


def video_html(path: str, max_height: int = 600) -> str:
    mime, _ = mimetypes.guess_type(path)
    mime = mime or "video/mp4"
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("ascii")
    return (
        f'<video controls style="max-width:100%;max-height:{max_height}px;" '
        f'src="data:{mime};base64,{data}"></video>'
    )


def run_concat(video_paths_text, ann_count, dropdowns, texts):
    if not video_paths_text.strip():
        return None, "请输入至少一个视频路径", []

    paths = [p.strip() for p in video_paths_text.strip().split("\n") if p.strip()]
    validated = []
    for p in paths:
        norm_path, err = validate_path(p)
        if err:
            return None, f"路径校验失败: {err}", []
        validated.append(norm_path)

    if not validated:
        return None, "没有有效视频", []

    n = len(validated)
    audio_flags = [has_audio_stream(v) for v in validated]

    per_video = [[] for _ in range(n)]
    for i in range(int(ann_count)):
        position = dropdowns[i] or "top"
        block = texts[i] or ""
        lines = block.split("\n")
        for vid_idx in range(n):
            raw = lines[vid_idx] if vid_idx < len(lines) else ""
            stripped, color = parse_color_prefix(raw)
            stripped = stripped.strip()
            if not stripped:
                continue
            srt_match = _SRT_TOKEN_RE.match(stripped)
            if srt_match:
                srt_path, err = validate_subtitle_path(srt_match.group(1))
                if err:
                    return None, f"字幕校验失败: {err}", []
                per_video[vid_idx].append(
                    {"kind": "srt", "path": srt_path, "position": position, "color": color}
                )
            else:
                per_video[vid_idx].append(
                    {"kind": "text", "position": position, "text": stripped, "color": color}
                )

    try:
        filter_parts = []
        for vid_idx in range(n):
            chain = f"[{vid_idx}:v]"
            anns = per_video[vid_idx]
            filters = []
            for ann in anns:
                if ann["kind"] == "srt":
                    filters.append(build_subtitles(ann["path"], ann["position"], ann["color"]))
                else:
                    expanded = expand_annotation(ann["text"], validated[vid_idx])
                    filters.append(build_drawtext(expanded, ann["position"], ann["color"]))
            chain += ",".join(filters) if filters else "null"
            chain += f"[vann{vid_idx}]"
            filter_parts.append(chain)

        # 单个视频无需 hstack（hstack 要求 inputs>=2），直接输出带 annotation 的画面；
        # 多个视频则横向拼接。
        if n >= 2:
            hstack_inputs = "".join(f"[vann{i}]" for i in range(n))
            filter_parts.append(f"{hstack_inputs}hstack=inputs={n}[v]")
        else:
            filter_parts.append("[vann0]null[v]")

        audio_idxs = [i for i, has in enumerate(audio_flags) if has]
        if len(audio_idxs) >= 2:
            # 每路先统一成立体声，再用 amix 混音。
            # 不用 amerge：amerge 会把各路声道拼接叠加（N 路 stereo -> 2N 声道），
            # 产生 AAC 不支持的非标准声道布局（如 10 声道）而编码失败。
            # 不用 amix 的 normalize 选项（旧版 ffmpeg 不支持），改为混音后用
            # volume 手动补偿归一化带来的音量衰减。
            aud_prep = "".join(
                f"[{i}:a]aformat=channel_layouts=stereo[a{i}];" for i in audio_idxs
            )
            audio_inputs = "".join(f"[a{i}]" for i in audio_idxs)
            k = len(audio_idxs)
            filter_parts.append(
                f"{aud_prep}{audio_inputs}"
                f"amix=inputs={k},volume={k}[a]"
            )
            audio_map = ["-map", "[a]"]
        elif len(audio_idxs) == 1:
            audio_map = ["-map", f"{audio_idxs[0]}:a"]
        else:
            audio_map = []

        filter_complex = ";".join(filter_parts)

        output_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False, dir="/tmp").name

        cmd = ["ffmpeg", "-y"]
        for v in validated:
            cmd += ["-i", v]
        cmd += [
            "-filter_complex", filter_complex,
            "-map", "[v]",
            *audio_map,
            "-c:v", "libx264", "-preset", "fast", "-crf", "23",
        ]
        if audio_map:
            cmd += ["-c:a", "aac", "-b:a", "192k"]
        cmd += [output_path]

        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            return None, f"FFmpeg 错误:\n{result.stderr[-2000:]}", []

        previews = []
        for vid_idx in range(n):
            previews.append((validated[vid_idx], per_video[vid_idx]))

        audio_note = ""
        if not any(audio_flags):
            audio_note = "（所有输入均无音频，输出无音轨）"
        elif not all(audio_flags):
            missing = [os.path.basename(validated[i]) for i, f in enumerate(audio_flags) if not f]
            audio_note = f"（部分视频无音频：{', '.join(missing)}，已跳过）"

        if n == 1:
            done_msg = f"成功处理 1 个视频，应用 {int(ann_count)} 个 annotation{audio_note}"
        else:
            done_msg = f"成功拼接 {n} 个视频，应用 {int(ann_count)} 个 annotation{audio_note}"

        return (output_path, done_msg, previews)

    except Exception as e:
        return None, f"处理异常: {e}", []

In [ ]:
path_input = widgets.Textarea(
    description="视频路径",
    placeholder="每行一个，例如:\n/path/to/video1.mp4\n/path/to/video2.mp4",
    layout=widgets.Layout(width="100%", height="120px"),
    style={"description_width": "80px"},
)
concat_btn = widgets.Button(description="拼接", button_style="primary", icon="play")
reset_btn = widgets.Button(description="重置", button_style="danger", icon="times")

ann_count_state = {"value": 0}
ann_rows = []
ann_dropdowns = []
ann_texts = []

for i in range(MAX_ANNOTATIONS):
    dd = widgets.Dropdown(
        options=POSITIONS, value=POSITIONS[i] if i < len(POSITIONS) else POSITIONS[0],
        description=f"#{i+1} 位置",
        layout=widgets.Layout(width="220px"),
        style={"description_width": "60px"},
    )
    tb = widgets.Textarea(
        description=f"#{i+1} 文本",
        placeholder=(
            "第1个视频的文本\n第2个视频的文本\n...\n"
            "支持转义：\\frame_idx 当前帧编号，\\name 视频文件名，\\timestamp 秒表\n"
            "行首加 \\red \\blue \\yellow 可改文字颜色（默认白色）\n"
            "行内用 \\srt /path/to/sub.srt 烧录字幕文件"
        ),
        layout=widgets.Layout(width="100%", height="70px"),
        style={"description_width": "60px"},
    )
    row = widgets.HBox(
        [dd, tb],
        layout=widgets.Layout(display="none", width="100%", margin="4px 0"),
    )
    ann_rows.append(row)
    ann_dropdowns.append(dd)
    ann_texts.append(tb)

add_btn = widgets.Button(description="+ 新增 annotation")
remove_btn = widgets.Button(description="- 删除最后一个")

_updating_options = {"flag": False}


def refresh_dropdown_options(_change=None):
    if _updating_options["flag"]:
        return
    _updating_options["flag"] = True
    try:
        count = ann_count_state["value"]
        active_vals = [ann_dropdowns[i].value for i in range(count)]
        for i in range(count):
            current = ann_dropdowns[i].value
            used_by_others = {v for j, v in enumerate(active_vals) if j != i and v is not None}
            allowed = [p for p in POSITIONS if p == current or p not in used_by_others]
            if list(ann_dropdowns[i].options) != allowed:
                ann_dropdowns[i].options = allowed
                ann_dropdowns[i].value = current
    finally:
        _updating_options["flag"] = False


for dd in ann_dropdowns:
    dd.observe(refresh_dropdown_options, names="value")


def refresh_rows():
    for i, row in enumerate(ann_rows):
        row.layout.display = "flex" if i < ann_count_state["value"] else "none"
    refresh_dropdown_options()


def _on_add(_):
    if ann_count_state["value"] >= MAX_ANNOTATIONS:
        return
    next_idx = ann_count_state["value"]
    used = {ann_dropdowns[i].value for i in range(next_idx)}
    default = next((p for p in POSITIONS if p not in used), POSITIONS[0])
    _updating_options["flag"] = True
    try:
        ann_dropdowns[next_idx].options = POSITIONS
        ann_dropdowns[next_idx].value = default
    finally:
        _updating_options["flag"] = False
    ann_count_state["value"] = next_idx + 1
    refresh_rows()


def _on_remove(_):
    ann_count_state["value"] = max(ann_count_state["value"] - 1, 0)
    refresh_rows()


add_btn.on_click(_on_add)
remove_btn.on_click(_on_remove)

status_out = widgets.Output()
result_out = widgets.Output()
previews_out = widgets.Output()


def on_concat(_):
    with status_out:
        clear_output()
        print("处理中…")
    with result_out:
        clear_output()
    with previews_out:
        clear_output()

    dd_vals = [d.value for d in ann_dropdowns]
    tx_vals = [t.value for t in ann_texts]
    out_path, msg, previews = run_concat(
        path_input.value, ann_count_state["value"], dd_vals, tx_vals,
    )

    with status_out:
        clear_output()
        color = "red" if out_path is None else "green"
        display(HTML(f'<div style="color:{color};white-space:pre-wrap;">{msg}</div>'))

    if out_path:
        with result_out:
            display(HTML("<h3 style='text-align:center;'>拼接结果</h3>"))
            mime, _ = mimetypes.guess_type(out_path)
            mime = mime or "video/mp4"
            with open(out_path, "rb") as f:
                data = base64.b64encode(f.read()).decode("ascii")
            display(HTML(
                f'<div style="display:flex;justify-content:center;width:100%;">'
                f'<video controls style="max-width:100%;max-height:600px;" '
                f'src="data:{mime};base64,{data}"></video>'
                f'</div>'
            ))

        with previews_out:
            display(HTML("<h3>各视频预览</h3>"))
            for idx, (vp, anns) in enumerate(previews):
                if anns:
                    items = []
                    for ann in anns:
                        col = ann.get("color")
                        style = f"color:{COLOR_HTML[col]};" if col else ""
                        if ann["kind"] == "srt":
                            label = f"srt/{ann['position']}"
                            content = os.path.basename(ann["path"])
                        else:
                            label = ann["position"]
                            content = ann["text"]
                        items.append(
                            f"<li><b>[{label}]</b> <span style='{style}'>{content}</span></li>"
                        )
                    ann_html = "".join(items)
                else:
                    ann_html = "<li><i>(无)</i></li>"
                display(HTML(
                    f'<div style="display:flex;gap:16px;align-items:flex-start;'
                    f'border-top:1px solid #ddd;padding:12px 0;">'
                    f'<div style="flex:0 0 320px;">'
                    f'<video controls style="width:320px;" src="data:video/mp4;base64,'
                    f'{base64.b64encode(open(vp, "rb").read()).decode("ascii")}"></video>'
                    f'</div>'
                    f'<div style="flex:1;">'
                    f'<div><b>视频 {idx+1}</b></div>'
                    f'<div style="word-break:break-all;"><b>路径:</b> {vp}</div>'
                    f'<div><b>Annotations:</b><ul>{ann_html}</ul></div>'
                    f'</div>'
                    f'</div>'
                ))


concat_btn.on_click(on_concat)


def on_reset(_):
    path_input.value = ""
    _updating_options["flag"] = True
    try:
        for i, dd in enumerate(ann_dropdowns):
            dd.options = POSITIONS
            dd.value = POSITIONS[i] if i < len(POSITIONS) else POSITIONS[0]
        for tb in ann_texts:
            tb.value = ""
    finally:
        _updating_options["flag"] = False
    ann_count_state["value"] = 0
    refresh_rows()
    with status_out:
        clear_output()
    with result_out:
        clear_output()
    with previews_out:
        clear_output()


reset_btn.on_click(on_reset)

ui = widgets.VBox([
    widgets.HBox(
        [path_input, widgets.VBox([concat_btn, reset_btn])],
        layout=widgets.Layout(align_items="flex-start"),
    ),
    widgets.HTML(
        "<h3 style='margin-bottom:4px;'>Annotations</h3>"
        "<div style='color:#666;font-size:12px;margin-bottom:6px;'>"
        "文本中可使用转义：<code>\\frame_idx</code> 显示当前帧编号（动态），"
        "<code>\\name</code> 显示视频文件名，"
        "<code>\\timestamp</code> 显示秒表（当前播放时间 HH:MM:SS.mmm，动态）。<br>"
        "行首加 <code style='color:#FF6B6B;'>\\red</code> "
        "<code style='color:#5BC0EB;'>\\blue</code> "
        "<code style='color:#F7E967;background:#333;padding:0 2px;'>\\yellow</code> "
        "可设置该行文字颜色，未指定则使用默认白色。<br>"
        "行内用 <code>\\srt /path/to/sub.srt</code> 为该视频按时间轴烧录字幕"
        "（支持 .srt/.ass/.ssa/.vtt，位置由左侧下拉框控制，颜色前缀同样生效）。"
        "</div>"
    ),
    widgets.HBox([add_btn, remove_btn]),
    widgets.VBox(ann_rows),
    status_out,
    result_out,
    previews_out,
])

display(ui)